
# Notebook 16 · Activación local y store marketing con stock disponible

## Pregunta de negocio
¿Cómo activar campañas hiperlocales por tienda y categoría usando afinidad del cliente y stock real?

## Qué aprenderás
- Cómo conectar cercanía, afinidad, visita probable y disponibilidad.
- Cómo construir un score de capturabilidad local.
- Cómo pasar de datos a un plan por tienda.

## Outputs
- `nb16_local_activation_plan.csv`
- `nb16_store_category_summary.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

afinidad = tables["afinidad"].copy()
stock = tables["stock"].copy()
fact = tables["fact"].copy()
productos = tables["productos"].copy()
tiendas = tables["tiendas"].copy()

sales = fact[(fact["tipo_movimiento"] == "VENTA") & fact["customer_id"].notna()].copy()

cat_aff = (
    sales.groupby(["customer_id", "categoria"])
         .agg(revenue=("importe_abs_num", "sum"), tx=("doc_id_std", "nunique"))
         .reset_index()
)
cat_aff["cat_affinity_score"] = cat_aff["revenue"] * 0.7 + cat_aff["tx"] * 50

top_cat = (
    cat_aff.sort_values(["customer_id", "cat_affinity_score"], ascending=[True, False])
           .groupby("customer_id")
           .head(2)
)

latest_stock = stock.sort_values("fecha_dt").groupby(["store_id", "product_id"]).tail(1).copy()
latest_stock["available_units"] = latest_stock["stock_cierre"].fillna(0) - latest_stock["stock_reservado"].fillna(0)
latest_stock = latest_stock.merge(
    productos[["product_id", "categoria", "nombre_producto", "precio_venta_num", "coste_unitario_num"]],
    on="product_id", how="left"
)

store_category = (
    latest_stock.groupby(["store_id", "categoria"])
                .agg(
                    skus_disponibles=("product_id", "nunique"),
                    units_disponibles=("available_units", "sum"),
                    precio_mediano=("precio_venta_num", "median"),
                )
                .reset_index()
)

store_category.head(10)


,store_id,categoria,skus_disponibles,units_disponibles,precio_mediano
0,ONLINE,Decoración,1,15.0,77.19
1,ONLINE,Iluminación,1,2.0,192.10
2,ONLINE,Muebles,1,15.0,221.99
3,ONLINE,Textil hogar,1,53.0,218.54
4,ONLINE,Decoración,31,2252.0,144.85
5,ONLINE,Iluminación,28,1838.0,201.03
6,ONLINE,Muebles,21,745.0,733.00
7,ONLINE,Textil hogar,18,1044.0,101.43
8,ONLINE,decoración,1,136.0,1131.94
9,S001,Decoración,26,1589.0,122.73


In [3]:

rows = []
base = afinidad[afinidad["es_capturable_localmente"] == 1].copy()

for _, r in base.iterrows():
    cust = r["customer_id"]
    store = r["store_id_mas_cercana"]
    cats = top_cat[top_cat["customer_id"] == cust]
    if cats.empty:
        continue

    for _, c in cats.iterrows():
        sc = store_category[(store_category["store_id"] == store) & (store_category["categoria"] == c["categoria"])]
        if sc.empty:
            continue
        sc = sc.iloc[0]

        targetability_score = (
            (r["probabilidad_visita_30d"] * 0.50)
            + (min(sc["skus_disponibles"] / 20, 1) * 0.20)
            + (min(c["cat_affinity_score"] / 2000, 1) * 0.30)
        ) * (1 - min(r["distancia_km_store_mas_cercana"] / 50, 0.50))

        rows.append({
            "customer_id": cust,
            "store_id_target": store,
            "categoria_target": c["categoria"],
            "prob_visita_30d": r["probabilidad_visita_30d"],
            "distance_km": r["distancia_km_store_mas_cercana"],
            "cat_affinity_score": c["cat_affinity_score"],
            "skus_disponibles": sc["skus_disponibles"],
            "units_disponibles": sc["units_disponibles"],
            "targetability_score": targetability_score,
        })

local_plan = pd.DataFrame(rows).merge(
    tiendas[["store_id", "nombre_tienda", "ciudad", "provincia", "region"]],
    left_on="store_id_target", right_on="store_id", how="left"
)

local_plan["accion_sugerida"] = np.where(
    local_plan["targetability_score"] >= local_plan["targetability_score"].quantile(0.75),
    "Campaña local prioritaria con stock destacado",
    "Nurture local con contenido y visita a tienda"
)

local_plan.sort_values("targetability_score", ascending=False).head(20)


,customer_id,store_id_target,categoria_target,prob_visita_30d,distance_km,cat_affinity_score,skus_disponibles,units_disponibles,targetability_score,store_id,nombre_tienda,ciudad,provincia,region,accion_sugerida
180,C0092,S001,Muebles,0.436,1.7,1637.508,20,469.000,0.641063,S001,FraSoHome Madrid Centro,Madrid,Madrid,Centro,Campaña local prioritaria con stock destacado
148,C0076,S002,Muebles,0.294,2.8,3697.424,22,526.000,0.610768,S002,FraSoHome Barcelona Eixample,Barcelona,Barcelona,Cataluña,Campaña local prioritaria con stock destacado
149,C0076,S002,Decoración,0.294,2.8,1938.314,28,1609.000,0.602033,S002,FraSoHome Barcelona Eixample,Barcelona,Barcelona,Cataluña,Campaña local prioritaria con stock destacado
90,C0046,S002,Decoración,0.450,9.2,2297.581,28,1609.000,0.591600,S002,FraSoHome Barcelona Eixample,Barcelona,Barcelona,Cataluña,Campaña local prioritaria con stock destacado
89,C0046,S002,Muebles,0.450,9.2,2429.101,22,526.000,0.591600,S002,FraSoHome Barcelona Eixample,Barcelona,Barcelona,Cataluña,Campaña local prioritaria con stock destacado
146,C0075,S005,Muebles,0.381,6.0,3052.003,18,279.113,0.590040,S005,FraSoHome Bilbao Abando,Bilbao,Bizkaia,Norte,Campaña local prioritaria con stock destacado
188,C0096,S002,Iluminación,0.367,2.7,1516.272,25,1278.000,0.577950,S002,FraSoHome Barcelona Eixample,Barcelona,Barcelona,Cataluña,Campaña local prioritaria con stock destacado
147,C0075,S005,Decoración,0.381,6.0,1773.884,30,1784.000,0.577793,S005,FraSoHome Bilbao Abando,Bilbao,Bizkaia,Norte,Campaña local prioritaria con stock destacado
24,C0013,S001,Muebles,0.231,3.2,2050.452,20,469.000,0.576108,S001,FraSoHome Madrid Centro,Madrid,Madrid,Centro,Campaña local prioritaria con stock destacado
181,C0092,S001,Decoración,0.436,1.7,1185.103,26,1589.000,0.575509,S001,FraSoHome Madrid Centro,Madrid,Madrid,Centro,Campaña local prioritaria con stock destacado


In [4]:

store_summary = (
    local_plan.groupby(["store_id_target", "nombre_tienda", "categoria_target"])
              .agg(
                  clientes_target=("customer_id", "nunique"),
                  score_medio=("targetability_score", "mean"),
                  prob_visita_media=("prob_visita_30d", "mean"),
                  stock_total=("units_disponibles", "sum"),
              )
              .reset_index()
              .sort_values(["clientes_target", "score_medio"], ascending=[False, False])
)

store_summary.head(20)


,store_id_target,nombre_tienda,categoria_target,clientes_target,score_medio,prob_visita_media,stock_total
18,S005,FraSoHome Bilbao Abando,Muebles,22,0.379002,0.256091,6140.486
7,S002,FraSoHome Barcelona Eixample,Muebles,15,0.388389,0.249600,7890.000
11,S003,FraSoHome Valencia Ruzafa,Muebles,14,0.402697,0.272286,7056.000
15,S004,FraSoHome Sevilla Nervión,Muebles,14,0.367765,0.272857,5026.000
16,S005,FraSoHome Bilbao Abando,Decoración,13,0.373787,0.295462,23192.000
17,S005,FraSoHome Bilbao Abando,Iluminación,13,0.359152,0.269308,14976.000
2,S001,FraSoHome Madrid Centro,Muebles,11,0.447089,0.293182,5159.000
5,S002,FraSoHome Barcelona Eixample,Decoración,11,0.391248,0.261818,17699.000
10,S003,FraSoHome Valencia Ruzafa,Iluminación,10,0.395584,0.275900,8160.000
9,S003,FraSoHome Valencia Ruzafa,Decoración,9,0.416681,0.266667,15072.741



## Lectura de negocio

El marketing local deja de ser una campaña “por tienda” y pasa a ser una activación por:

- cliente capturable,
- categoría de afinidad,
- tienda con stock suficiente,
- y probabilidad de visita.

Es exactamente la lógica de marketing de precisión aplicada a retail omnicanal.


In [5]:

path_plan = mpu.export_csv(local_plan.sort_values("targetability_score", ascending=False), "nb16_local_activation_plan.csv")
path_summary = mpu.export_csv(store_summary, "nb16_store_category_summary.csv")

print("Guardado:", path_plan)
print("Guardado:", path_summary)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb16_local_activation_plan.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb16_store_category_summary.csv
